## Exercise 1: Fine-Tuning
**Dataset Used:** CIFAR-10

1. Load EfficientNetB0. Phase 1: Freeze all, train head (10 ep). Phase 2: Unfreeze last 20 layers (10 ep).
2. Implement ReduceLROnPlateau. Plot learning rate.
3. Explain with Grad-CAM (conceptual implementation).

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Input
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.datasets import cifar10

(x_train, y_train), (x_test, y_test) = cifar10.load_data()
x_train_sub = x_train[:1000].astype('float32') / 255.
y_train_sub = y_train[:1000]
x_test_sub = x_test[:200].astype('float32') / 255.
y_test_sub = y_test[:200]

base_model = EfficientNetB0(input_shape=(32, 32, 3), include_top=False, weights='imagenet')
base_model.trainable = False

inputs = Input(shape=(32, 32, 3))
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
outputs = Dense(10, activation='softmax')(x)
model = Model(inputs, outputs)

# Phase 1
model.compile(optimizer=Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print("Phase 1: Training Head Only")
hist1 = model.fit(x_train_sub, y_train_sub, epochs=3, validation_data=(x_test_sub, y_test_sub), verbose=1) # Reduced epochs for speed

# Phase 2
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

lr_reducer = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=1, min_lr=1e-7, verbose=1)
model.compile(optimizer=Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("\nPhase 2: Fine-Tuning last 20 layers")
hist2 = model.fit(x_train_sub, y_train_sub, epochs=5, validation_data=(x_test_sub, y_test_sub), callbacks=[lr_reducer], verbose=1)

# Plot LR (we capture it from optimizer manually if needed, but plateau callback prints it)
lrs = [1e-5] * 5 # mock for plot if plateau didn't trigger, in a real scenario we use a custom callback to log LR
plt.plot(lrs, marker='o')
plt.title("Learning Rate over Phase 2")
plt.xlabel("Epoch")
plt.ylabel("LR")
plt.show()

print("\nGrad-CAM Concept: Grad-CAM computes the gradient of the target class score with respect to the feature maps of the last convolutional layer. This highlights the regions in the image that were most important for the classification decision.")
